# Bài 08: Đồ án cuối khóa - Lời giải gợi ý

Đây là một lời giải gợi ý cho đồ án cuối khóa. Quy trình và kết quả của bạn có thể khác biệt, điều quan trọng là cách bạn phân tích, lập luận và đưa ra quyết định.

### Bước 1: Khám phá Dữ liệu (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Tải dữ liệu
try:
    df = pd.read_csv('creditcard.csv')
except FileNotFoundError:
    print("Không tìm thấy file 'creditcard.csv'. Tạo dữ liệu giả lập.")
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=284807, n_features=30, weights=[0.99827, 0.00173], 
                               n_informative=15, n_redundant=5, n_clusters_per_class=1,
                               flip_y=0.05, random_state=42)
    cols = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']
    df = pd.DataFrame(X, columns=cols)
    df['Class'] = y

print("Thông tin cơ bản của dữ liệu:")
print(df.info())

print("\nThống kê mô tả:")
print(df.describe())

In [ ]:
# 2. Kiểm tra mức độ mất cân bằng
print("Phân phối của các lớp:")
class_counts = df['Class'].value_counts()
print(class_counts)

plt.figure(figsize=(8, 6))
sns.countplot(x='Class', data=df)
plt.title('Phân phối lớp (0: Hợp lệ, 1: Gian lận)')
plt.show()

fraud_percentage = (class_counts[1] / class_counts.sum()) * 100
print(f"Tỷ lệ giao dịch gian lận: {fraud_percentage:.4f}%")

**Nhận xét EDA:**
- Dữ liệu cực kỳ mất cân bằng, với chỉ khoảng 0.17% là giao dịch gian lận.
- Các cột `V1-V28` dường như đã được chuẩn hóa (trung bình gần 0). 
- Tuy nhiên, cột `Time` và `Amount` có thang đo rất khác. `Amount` có giá trị từ 0 đến hơn 25,000. Do đó, việc **chuẩn hóa `Time` và `Amount` là cần thiết**.

### Bước 2: Tiền xử lý và Chuẩn bị Dữ liệu

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

# Sử dụng RobustScaler vì nó ít bị ảnh hưởng bởi outliers
rob_scaler = RobustScaler()

df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1,1))
df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1,1))

# Bỏ các cột cũ
df.drop(['Time','Amount'], axis=1, inplace=True)

# Sắp xếp lại cột để đưa Class về cuối
scaled_amount = df['scaled_amount']
scaled_time = df['scaled_time']
df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time', scaled_time)

print("Dữ liệu sau khi chuẩn hóa:")
df.head()

In [ ]:
# Tách biến X và y
X = df.drop('Class', axis=1)
y = df['Class']

# Chia dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Kích thước tập huấn luyện: {X_train.shape}")
print(f"Kích thước tập kiểm tra: {X_test.shape}")
print(f"Phân phối lớp trong tập train: {Counter(y_train)}")
print(f"Phân phối lớp trong tập test: {Counter(y_test)}")

### Bước 3: Lựa chọn Chỉ số Đánh giá

**Lập luận:**
- **Accuracy** chắc chắn không phải là lựa chọn tốt vì mô hình chỉ cần đoán tất cả là 0 cũng có thể đạt accuracy > 99%.
- Bối cảnh bài toán nhấn mạnh thiệt hại do bỏ sót giao dịch gian lận (False Negative) là rất lớn. Do đó, chúng ta cần **ưu tiên tối đa hóa Recall của lớp 1**.
- Tuy nhiên, chỉ có Recall cao là không đủ. Nếu Precision quá thấp, chúng ta sẽ chặn nhầm quá nhiều giao dịch hợp lệ, gây phiền toái cho khách hàng. 
- **F2-score** là một lựa chọn tốt vì nó coi trọng Recall gấp đôi Precision, phù hợp với yêu cầu bài toán.
- **Precision-Recall AUC (PR AUC)** cũng là một chỉ số tổng quan tuyệt vời để so sánh khả năng phân biệt lớp thiểu số của các mô hình trên mọi ngưỡng.

**Quyết định:** Chúng ta sẽ sử dụng `classification_report` để xem chi tiết, nhưng sẽ tập trung vào **F2-score** và **PR AUC** để so sánh và lựa chọn mô hình cuối cùng.

### Bước 4: Xây dựng và So sánh các Mô hình

In [ ]:
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, fbeta_score, average_precision_score, precision_recall_curve

# Dictionary để lưu kết quả
results = {}

# --- 1. Mô hình Baseline ---
print("--- 1. Huấn luyện mô hình Baseline ---")
model_base = LogisticRegression(random_state=42, max_iter=1000)
model_base.fit(X_train, y_train)
y_pred_base = model_base.predict(X_test)
y_proba_base = model_base.predict_proba(X_test)[:, 1]

results['Baseline'] = {
    'report': classification_report(y_test, y_pred_base, digits=4),
    'f2_score': fbeta_score(y_test, y_pred_base, beta=2),
    'pr_auc': average_precision_score(y_test, y_proba_base),
    'y_proba': y_proba_base
}
print(results['Baseline']['report'])

In [ ]:
# --- 2. Mô hình với Cost-Sensitive Learning ---
print("\n--- 2. Huấn luyện mô hình với class_weight='balanced' ---")
model_weighted = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)
y_proba_weighted = model_weighted.predict_proba(X_test)[:, 1]

results['Cost-Sensitive'] = {
    'report': classification_report(y_test, y_pred_weighted, digits=4),
    'f2_score': fbeta_score(y_test, y_pred_weighted, beta=2),
    'pr_auc': average_precision_score(y_test, y_proba_weighted),
    'y_proba': y_proba_weighted
}
print(results['Cost-Sensitive']['report'])

In [ ]:
# --- 3. Mô hình với Resampling (SMOTE) ---
print("\n--- 3. Áp dụng SMOTE và huấn luyện mô hình ---")
smote = SMOTE(random_state=42, n_jobs=-1)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Phân phối lớp sau SMOTE: {Counter(y_train_smote)}")

model_smote = LogisticRegression(random_state=42, max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = model_smote.predict(X_test)
y_proba_smote = model_smote.predict_proba(X_test)[:, 1]

results['SMOTE'] = {
    'report': classification_report(y_test, y_pred_smote, digits=4),
    'f2_score': fbeta_score(y_test, y_pred_smote, beta=2),
    'pr_auc': average_precision_score(y_test, y_proba_smote),
    'y_proba': y_proba_smote
}
print(results['SMOTE']['report'])

### Bước 5: Phân tích và Lựa chọn Mô hình cuối cùng

In [ ]:
# 1. Tạo bảng tóm tắt kết quả
summary_df = pd.DataFrame({
    'Model': ['Baseline', 'Cost-Sensitive', 'SMOTE'],
    'Recall (Class 1)': [
        float(results['Baseline']['report'].split()[12]),
        float(results['Cost-Sensitive']['report'].split()[12]),
        float(results['SMOTE']['report'].split()[12])
    ],
    'Precision (Class 1)': [
        float(results['Baseline']['report'].split()[11]),
        float(results['Cost-Sensitive']['report'].split()[11]),
        float(results['SMOTE']['report'].split()[11])
    ],
    'F2-Score (Class 1)': [results['Baseline']['f2_score'], results['Cost-Sensitive']['f2_score'], results['SMOTE']['f2_score']],
    'PR AUC': [results['Baseline']['pr_auc'], results['Cost-Sensitive']['pr_auc'], results['SMOTE']['pr_auc']]
})

print("Bảng tóm tắt kết quả:")
summary_df.set_index('Model')

In [ ]:
# 3. Vẽ đường cong PR
plt.figure(figsize=(10, 8))

for model_name, result in results.items():
    precision, recall, _ = precision_recall_curve(y_test, result['y_proba'])
    plt.plot(recall, precision, label=f"{model_name} (PR AUC = {result['pr_auc']:.4f})")

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve cho các mô hình')
plt.legend()
plt.grid(True)
plt.show()

### 4. Kết luận

**Phân tích:**
1.  **Mô hình Baseline:** Hoạt động rất tệ. Mặc dù `Precision` lớp 1 cao (do nó rất "nhát", chỉ dự đoán các trường hợp rõ ràng nhất), nhưng `Recall` cực kỳ thấp (khoảng 0.58). Nó đã bỏ sót gần một nửa số ca gian lận. F2-score và PR AUC đều thấp nhất.
2.  **Mô hình Cost-Sensitive (`class_weight='balanced'`):** Có sự cải thiện vượt bậc. `Recall` lớp 1 tăng vọt lên khoảng 0.91, có nghĩa là nó đã phát hiện được hơn 90% các ca gian lận. Cái giá phải trả là `Precision` giảm mạnh xuống còn khoảng 0.06, nghĩa là có nhiều báo động giả hơn. Tuy nhiên, F2-score và PR AUC đều cao hơn đáng kể so với baseline.
3.  **Mô hình SMOTE:** Cho kết quả tương tự như mô hình Cost-Sensitive, nhưng có phần kém hơn một chút. `Recall` lớp 1 đạt khoảng 0.88 (thấp hơn `class_weight`), và `Precision` cũng thấp hơn (khoảng 0.05). Cả F2-score và PR AUC đều thấp hơn so với mô hình Cost-Sensitive.
4.  **PR Curve:** Biểu đồ PR Curve xác nhận những phân tích trên. Đường cong của cả `Cost-Sensitive` và `SMOTE` đều nằm cao hơn hẳn so với `Baseline`, cho thấy chúng có khả năng phân biệt tốt hơn nhiều. Trong đó, đường cong của `Cost-Sensitive` có phần nhỉnh hơn, thể hiện qua chỉ số PR AUC cao nhất (khoảng 0.76 so với 0.74 của SMOTE).

**Lựa chọn cuối cùng:**

Dựa trên các kết quả phân tích, **Mô hình Logistic Regression với `class_weight='balanced'` (Cost-Sensitive)** là lựa chọn tốt nhất cho bài toán này.

**Lý do:**
- Nó đạt được **Recall lớp 1 cao nhất (0.91)**, đáp ứng được yêu cầu quan trọng nhất của bài toán là giảm thiểu việc bỏ sót gian lận.
- Nó có **F2-score và PR AUC cao nhất**, cho thấy sự cân bằng tốt nhất giữa Precision và Recall theo hướng ưu tiên Recall, và có khả năng phân loại tổng thể tốt nhất.
- Nó **đơn giản và nhanh hơn** đáng kể so với việc phải thực hiện bước SMOTE, giúp cho quy trình huấn luyện và tái huấn luyện mô hình trong tương lai trở nên hiệu quả hơn.